# 01: Data & Signals Exploration Notebook
Interactive exploration of price history, technical indicators (RSI, MACD, Bollinger Bands), and active signal detection.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / 'backend'))

import matplotlib.pyplot as plt
import pandas as pd
from app.database.db import SessionLocal, init_db
from app.ingestion.price_ingestion import get_price_history_dataframe, fetch_and_store_price_history
from app.signals.indicators import add_technical_indicators
from app.signals.signal_filter import detect_signals

init_db()
db = SessionLocal()

In [ ]:
symbol = 'RELIANCE.NS'
fetch_and_store_price_history(symbol, db, period='1y')
df = get_price_history_dataframe(symbol, db)
df_ind = add_technical_indicators(df)
print(f'Loaded {len(df_ind)} price history records for {symbol}')
df_ind.tail(5)

In [ ]:
# Plotting Price and Technical Indicators
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

ax1.plot(df_ind['date'], df_ind['close'], label='Close Price', color='#2563EB')
ax1.plot(df_ind['date'], df_ind['bb_upper'], '--', color='#93C5FD', label='BB Upper')
ax1.plot(df_ind['date'], df_ind['bb_lower'], '--', color='#93C5FD', label='BB Lower')
ax1.set_title(f'{symbol} Price History & Bollinger Bands')
ax1.legend()

ax2.plot(df_ind['date'], df_ind['rsi_14'], color='#7C3AED', label='RSI (14)')
ax2.axhline(70, linestyle='--', color='red', alpha=0.5)
ax2.axhline(30, linestyle='--', color='green', alpha=0.5)
ax2.set_title('RSI Indicator')
ax2.legend()

ax3.bar(df_ind['date'], df_ind['macd_hist'], color='#10B981', label='MACD Hist')
ax3.set_title('MACD Histogram')
ax3.legend()

plt.tight_layout()
plt.show()

In [ ]:
signals = detect_signals(df)
print(f'Active signals detected for {symbol}:')
for s in signals:
    print(f"- [{s['direction']}] {s['signal_type']} (Confidence: {s['confidence']*100:.0f}%): {s['description']}")